# Evaluate indication mapping

This notebook isolates the identity-mapping step. One LLM call maps existing MOAlmanac indications to indications from the latest label as `matched`, `new`, `not_found`, or `uncertain`.

Difference assessment, curator decisions about `same` versus `revised`, proposal generation, changelog assessment, and approval-date matching are intentionally out of scope here.

In [ ]:
import json
import os
from collections import Counter
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

from moalmanac_fda_curation.core.extract_indications_from_fda_label import (
    DEFAULT_MAX_TOKENS,
    DEFAULT_MODEL,
    extract_indications_from_label_url,
)
from moalmanac_fda_curation.core.reconcile_indications import (
    build_reconciliation_prompt,
    indexed_latest_indications,
    load_existing_indications,
    reconcile_indications,
)

## Configure the Opdivo comparison

The existing records come from MOAlmanac. The local Opdivo changelog is used only to identify the newest label URL; its events are not passed to the mapping LLM.

In [ ]:
env_path = find_dotenv(usecwd=True)
if not env_path:
    raise FileNotFoundError("No .env file found")
load_dotenv(env_path)
if not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError(f"ANTHROPIC_API_KEY is missing from {env_path}")

PROJECT_ROOT = Path(env_path).parent
WORKSPACE_ROOT = PROJECT_ROOT.parent
EXISTING_INDICATIONS_JSON = WORKSPACE_ROOT / "moalmanac-db/referenced/indications.json"
CHANGELOG_JSON = WORKSPACE_ROOT / "ai-assisted-gk-curation/analyses/fda-indications/extracted-indications/section1-changelogs/Opdivo-bla125554-section1-changelog.json"
EXTRACTION_DIR = PROJECT_ROOT / "analyses/reconciliation-inputs/opdivo-current-label"

changelog = json.loads(CHANGELOG_JSON.read_text())
latest_event = max(
    changelog["events"],
    key=lambda event: (event["date"], event["event_number"]),
)
LATEST_LABEL_URL = latest_event["label_url"]
print("Latest label date:", latest_event["date"])
print("Latest label URL:", LATEST_LABEL_URL)

## Optionally regenerate latest-label indications

Leave `REGENERATE = False` to evaluate mapping against the saved extraction. Set it to `True` only when you want to download the current label and rerun the indication-extraction LLM call.

In [ ]:
REGENERATE = False

if REGENERATE:
    extraction_paths = extract_indications_from_label_url(
        label_url=LATEST_LABEL_URL,
        brand_name="Opdivo",
        generic_name="nivolumab",
        application_number="BLA125554",
        labels_dir=EXTRACTION_DIR / "labels",
        indications_dir=EXTRACTION_DIR / "intermediate",
        model=DEFAULT_MODEL,
        max_tokens=DEFAULT_MAX_TOKENS,
        overwrite=True,
    )
else:
    extraction_paths = {
        "claude_chunked_indication_fields": (
            EXTRACTION_DIR
            / "intermediate/Opdivo-BLA125554-claude_chunked_indication_fields.json"
        )
    }

LATEST_EXTRACTION_JSON = extraction_paths["claude_chunked_indication_fields"]
print("Extraction artifact:", LATEST_EXTRACTION_JSON)

## Load and inspect the indication strings

Only indication IDs, extraction indexes, and indication strings are supplied to the mapping call. `biomarker_only=True` keeps this experiment aligned with the current MOAlmanac scope.

In [ ]:
existing_indications = load_existing_indications(
    EXISTING_INDICATIONS_JSON, document_id="doc:fda.opdivo"
)
latest_payload = json.loads(LATEST_EXTRACTION_JSON.read_text())
latest_indications = indexed_latest_indications(latest_payload, biomarker_only=True)

print(f"Existing indications: {len(existing_indications)}")
for indication in existing_indications:
    print(f"  {indication['id']}: {indication['indication']}")

print(f"\nLatest-label indications: {len(latest_indications)}")
for indication in latest_indications:
    print(
        f"  [{indication['latest_indication_index']}]: "
        f"{indication['indication']}"
    )

## Inspect the exact prompt

Read this before running the LLM. It is the complete identity-mapping prompt.

In [ ]:
mapping_prompt = build_reconciliation_prompt(
    existing_indications, latest_indications
)
print(mapping_prompt)

## Run one mapping call

This is the notebook's only LLM evaluation step. Verification checks that every indication is accounted for exactly once.

In [ ]:
reconciliation = reconcile_indications(
    existing_indications, latest_indications
)

print("Verified:", reconciliation["verified"])
print("Verification errors:", reconciliation["verification_errors"])
print(
    "Classification counts:",
    Counter(item["classification"] for item in reconciliation["mappings"]),
)

## Review every mapping

The paired strings are shown with the identity classification and mapping rationale.

In [ ]:
for number, mapping in enumerate(reconciliation["mappings"], start=1):
    existing = mapping["existing_indication"]
    latest = mapping["latest_indication"]
    print("=" * 100)
    print(f"MAPPING {number}: {mapping['classification'].upper()}")
    print("Existing ID:", mapping["existing_indication_id"])
    print("Latest index:", mapping["latest_indication_index"])
    print("Existing:", existing["indication"] if existing else None)
    print("Latest:  ", latest["indication"] if latest else None)
    print("Reason:  ", mapping["reason"])

## Optionally save this run

In [ ]:
SAVE_RESULTS = False
RESULTS_JSON = EXTRACTION_DIR / "opdivo-indication-mapping.json"

if SAVE_RESULTS:
    RESULTS_JSON.write_text(json.dumps(reconciliation, indent=2) + "\n")
    print("Saved:", RESULTS_JSON)